# MMF Step 5 — Post-Process and Evaluate
Reproducibility notebook for the **{use_case}** use case. Primary metric: **{metric}**.

In [ ]:
catalog = "{catalog}"
schema = "{schema}"
use_case = "{use_case}"
metric = "{metric}"  # smape, mape, mae, mse, rmse
print(f"Post-processing: {catalog}.{schema}, use case: {use_case}, metric: {metric}")

In [ ]:
# Verify forecast outputs exist
spark.sql(f"""
    SELECT 'evaluation' AS table_name, COUNT(*) AS row_count
    FROM {catalog}.{schema}.{use_case}_evaluation_output
    UNION ALL
    SELECT 'scoring', COUNT(*)
    FROM {catalog}.{schema}.{use_case}_scoring_output
""").show()

In [ ]:
# Model performance summary from evaluation output
spark.sql(f"""
    SELECT model,
           COUNT(DISTINCT unique_id) AS series_count,
           COUNT(*) AS total_rows,
           ROUND(AVG(metric_value), 4) AS avg_{metric}
    FROM {catalog}.{schema}.{use_case}_evaluation_output
    GROUP BY model
    ORDER BY avg_{metric} ASC
""").show(20)

In [ ]:
# Best model selection per series (lowest average metric wins)
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.{schema}.{use_case}_best_models AS
    SELECT eval.unique_id, eval.model, eval.avg_metric, score.ds, score.y,
           'main_pipeline' AS forecast_source
    FROM (
        SELECT unique_id, model, avg_metric,
               RANK() OVER (PARTITION BY unique_id ORDER BY avg_metric ASC) AS rank
        FROM (
            SELECT unique_id, model, AVG(metric_value) AS avg_metric
            FROM {catalog}.{schema}.{use_case}_evaluation_output
            GROUP BY unique_id, model
            HAVING AVG(metric_value) IS NOT NULL
        )
    ) AS eval
    INNER JOIN {catalog}.{schema}.{use_case}_scoring_output AS score
        ON eval.unique_id = score.unique_id AND eval.model = score.model
    WHERE eval.rank = 1
""")
print(f"best_models created: {catalog}.{schema}.{use_case}_best_models")

In [ ]:
# Model ranking by wins count
spark.sql(f"""
    SELECT model,
           COUNT(*) AS wins_count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS wins_pct,
           ROUND(AVG(avg_metric), 4) AS avg_{metric}
    FROM {catalog}.{schema}.{use_case}_best_models
    GROUP BY model
    ORDER BY wins_count DESC
""").show(20)

In [ ]:
# Create evaluation summary table
spark.sql(f"""
    CREATE OR REPLACE TABLE {catalog}.{schema}.{use_case}_evaluation_summary AS
    SELECT model,
           COUNT(*) AS wins_count,
           ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 2) AS wins_pct,
           ROUND(AVG(avg_metric), 4) AS avg_{metric}
    FROM {catalog}.{schema}.{use_case}_best_models
    GROUP BY model
    ORDER BY wins_count DESC
""")
print(f"evaluation_summary created: {catalog}.{schema}.{use_case}_evaluation_summary")

In [ ]:
# Overall forecast quality
spark.sql(f"""
    SELECT
        COUNT(DISTINCT unique_id) AS total_series,
        COUNT(DISTINCT model) AS distinct_winning_models,
        ROUND(AVG(avg_metric), 4) AS overall_avg_{metric}
    FROM {catalog}.{schema}.{use_case}_best_models
""").show()

In [ ]:
# Top 10 worst-performing series
spark.sql(f"""
    SELECT unique_id, model, ROUND(avg_metric, 4) AS avg_{metric}, forecast_source
    FROM {catalog}.{schema}.{use_case}_best_models
    WHERE avg_metric IS NOT NULL
    ORDER BY avg_metric DESC
    LIMIT 10
""").show(truncate=False)

In [ ]:
# Cross-reference with series profiling (skipped if profile table does not exist)
try:
    spark.sql(f"""
        SELECT b.model, p.forecastability_class,
               COUNT(*) AS series_count,
               ROUND(AVG(b.avg_metric), 4) AS avg_{metric}
        FROM {catalog}.{schema}.{use_case}_best_models b
        LEFT JOIN {catalog}.{schema}.{use_case}_series_profile p
            ON b.unique_id = p.unique_id
        GROUP BY b.model, p.forecastability_class
        ORDER BY p.forecastability_class, avg_{metric}
    """).show(40)
except Exception:
    print("Series profile table not found — skipping cross-reference.")